In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# -----------------------------
# Load Healthcare Dataset
# -----------------------------
data = load_breast_cancer()

X = data.data
y = data.target

# Normalize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

# -----------------------------
# Vertical Data Split
# -----------------------------
# Client A gets first 15 features
XA_train = X_train[:, :15]
XA_test = X_test[:, :15]

# Client B gets remaining features
XB_train = X_train[:, 15:]
XB_test = X_test[:, 15:]

# -----------------------------
# Client Models
# -----------------------------
class ClientModel(nn.Module):
    def __init__(self, input_size):
        super(ClientModel, self).__init__()
        self.fc = nn.Linear(input_size, 8)

    def forward(self, x):
        return torch.relu(self.fc(x))

# -----------------------------
# Server Model
# -----------------------------
class ServerModel(nn.Module):
    def __init__(self):
        super(ServerModel, self).__init__()
        self.fc = nn.Linear(16, 1)

    def forward(self, xa, xb):
        x = torch.cat((xa, xb), dim=1)
        return torch.sigmoid(self.fc(x))

# -----------------------------
# Initialize Models
# -----------------------------
clientA = ClientModel(15)
clientB = ClientModel(XB_train.shape[1])
server = ServerModel()

criterion = nn.BCELoss()

optA = optim.Adam(clientA.parameters(), lr=0.001)
optB = optim.Adam(clientB.parameters(), lr=0.001)
optS = optim.Adam(server.parameters(), lr=0.001)

# -----------------------------
# Training
# -----------------------------
epochs = 50

for epoch in range(epochs):

    # Local embeddings
    embedA = clientA(XA_train)
    embedB = clientB(XB_train)

    # Server prediction
    preds = server(embedA, embedB)

    loss = criterion(preds, y_train)

    optA.zero_grad()
    optB.zero_grad()
    optS.zero_grad()

    loss.backward()

    optA.step()
    optB.step()
    optS.step()

    if (epoch+1) % 10 == 0:
        print("Epoch:", epoch+1, "Loss:", loss.item())

# -----------------------------
# Testing
# -----------------------------
with torch.no_grad():

    embedA = clientA(XA_test)
    embedB = clientB(XB_test)

    preds = server(embedA, embedB)

    predicted = (preds > 0.5).float()

    accuracy = (predicted == y_test).float().mean()

print("Test Accuracy:", accuracy.item())

Epoch: 10 Loss: 0.6239247918128967
Epoch: 20 Loss: 0.5878966450691223
Epoch: 30 Loss: 0.5528271198272705
Epoch: 40 Loss: 0.517913818359375
Epoch: 50 Loss: 0.48309290409088135
Test Accuracy: 0.9561403393745422
